# S-DoT 유동인구 + 외국인 생활인구 복합분석

**목적**: 외국인 밀집도(TEMP_FOREIGNER)와 실제 유동인구(S-DoT WALK)를 결합하여 복합점수를 산출하고, Hub & Spoke 전략의 정밀도를 높인다.

**데이터 기간**: 2024.01 ~ 2025.09 (보고서 분석 기간과 일치)

In [ ]:
# 환경 설정 & 데이터 로드

import pandas as pd
import numpy as np
import os
import glob
import warnings
warnings.filterwarnings('ignore')

# 경로 설정
BASE = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '06_analysis', '01_foreigner'))
RAW_SDOT = os.path.join(BASE, '01_raw_data', 'S-DoT_WALK')
PROCESSED = os.path.join(BASE, '02_processed_data')

# 기존 외국인 분석 결과 로드
foreigner_path = os.path.join(PROCESSED, '외국인_자치구별_분석결과_전체.csv')
df_foreigner = pd.read_csv(foreigner_path)
print(f'[외국인 데이터] {len(df_foreigner)}개 자치구')
print(df_foreigner[['자치구', '일평균_총외국인']].head())

# S-DoT 전체 로드
sdot_files = sorted(glob.glob(os.path.join(RAW_SDOT, 'S-DoT_WALK_*.csv')))
print(f'\n[S-DoT] 로드할 파일 수: {len(sdot_files)}')

dfs = []
for f in sdot_files:
    for enc in ['utf-8', 'cp949', 'euc-kr']:
        try:
            df = pd.read_csv(f, encoding=enc)
            dfs.append(df)
            break
        except:
            continue

df_sdot_raw = pd.concat(dfs, ignore_index=True)
print(f'[S-DoT] 총 레코드 수: {len(df_sdot_raw):,}')
print(f'[S-DoT] 컬럼: {list(df_sdot_raw.columns)}')
df_sdot_raw.head(3)

In [ ]:
# S-DoT 데이터 전처리


# 영문 구명 → 한글 매핑
SDOT_GU_MAP = {
    'Jongno-gu': '종로구', 'Jung-gu': '중구', 'Yongsan-gu': '용산구',
    'Seongdong-gu': '성동구', 'Gwangjin-gu': '광진구', 'Dongdaemun-gu': '동대문구',
    'Jungnang-gu': '중랑구', 'Seongbuk-gu': '성북구', 'Gangbuk-gu': '강북구',
    'Dobong-gu': '도봉구', 'Nowon-gu': '노원구', 'Eunpyeong-gu': '은평구',
    'Seodaemun-gu': '서대문구', 'Mapo-gu': '마포구', 'Yangcheon-gu': '양천구',
    'Gangseo-gu': '강서구', 'Guro-gu': '구로구', 'Geumcheon-gu': '금천구',
    'Yeongdeungpo-gu': '영등포구', 'Dongjak-gu': '동작구', 'Gwanak-gu': '관악구',
    'Seocho-gu': '서초구', 'Gangnam-gu': '강남구', 'Songpa-gu': '송파구',
    'Gangdong-gu': '강동구'
}

df_sdot = df_sdot_raw.copy()

# 자치구 한글 매핑
df_sdot['자치구_한글'] = df_sdot['자치구'].map(SDOT_GU_MAP)

# 방문자수 숫자 변환
df_sdot['방문자수'] = pd.to_numeric(df_sdot['방문자수'], errors='coerce').fillna(0)

# 측정시간에서 날짜 및 시간 추출
df_sdot['측정일'] = df_sdot['측정시간'].str[:10]
df_sdot['시간'] = df_sdot['측정시간'].str[11:13].astype(int)

# 영업시간 필터 (10~22시)
df_sdot = df_sdot[df_sdot['시간'].between(10, 22)].copy()
print(f'[전처리] 영업시간(10~22시) 필터 후: {len(df_sdot):,} 레코드')

# 분석 기간 필터 (2024.01 ~ 2025.09)
df_sdot['측정일_dt'] = pd.to_datetime(df_sdot['측정일'], errors='coerce')
df_sdot = df_sdot[
    (df_sdot['측정일_dt'] >= '2024-01-01') &
    (df_sdot['측정일_dt'] <= '2025-09-30')
].copy()
print(f'[전처리] 기간 필터(2024.01~2025.09) 후: {len(df_sdot):,} 레코드')

# 매핑되지 않는 자치구 제거 (Seoul_Grand_Park 등)
unmapped = df_sdot[df_sdot['자치구_한글'].isna()]['자치구'].unique()
print(f'[전처리] 매핑 제외 자치구: {unmapped}')
df_sdot = df_sdot[df_sdot['자치구_한글'].notna()].copy()

# 분석 일수 계산
total_days = df_sdot['측정일'].nunique()
print(f'[전처리] 분석 일수: {total_days}일')
print(f'[전처리] 최종 레코드 수: {len(df_sdot):,}')

[전처리] 영업시간(10~22시) 필터 후: 4,903,100 레코드


[전처리] 기간 필터(2024.01~2025.09) 후: 4,412,706 레코드
[전처리] 매핑 제외 자치구: ['Seoul_Grand_Park']


[전처리] 분석 일수: 632일
[전처리] 최종 레코드 수: 4,038,703


In [ ]:
# 셀 3: S-DoT 자치구별 유동인구 분석


# 자치구별 집계
sdot_gu = df_sdot.groupby('자치구_한글').agg(
    방문자수_합=('방문자수', 'sum'),
    센서수=('시리얼', 'nunique'),
    레코드수=('방문자수', 'count')
).reset_index()

sdot_gu.rename(columns={'자치구_한글': '자치구'}, inplace=True)

# 일평균 방문자수
sdot_gu['일평균_방문자'] = (sdot_gu['방문자수_합'] / total_days).round(0)

# 센서당 평균방문자 (센서 수 대비 보정)
sdot_gu['센서당_일평균방문자'] = (sdot_gu['일평균_방문자'] / sdot_gu['센서수']).round(0)

sdot_gu = sdot_gu.sort_values('일평균_방문자', ascending=False)

print('=' * 65)
print('S-DoT 자치구별 유동인구 TOP 10 (일평균 방문자)')
print('=' * 65)
print(sdot_gu[['자치구', '일평균_방문자', '센서수', '센서당_일평균방문자']].head(10).to_string(index=False))
print(f'\n전체 {len(sdot_gu)}개 자치구')

S-DoT 자치구별 유동인구 TOP 10 (일평균 방문자)
자치구  일평균_방문자  센서수  센서당_일평균방문자
강남구 105765.0    9     11752.0
동작구 100824.0    5     20165.0
 중구  93822.0    6     15637.0
양천구  90469.0    5     18094.0
구로구  81066.0    5     16213.0
도봉구  63945.0    6     10658.0
강동구  60684.0    7      8669.0
광진구  54661.0    5     10932.0
강북구  52272.0    5     10454.0
금천구  50587.0    4     12647.0

전체 22개 자치구


In [ ]:
# S-DoT 지역유형별 분석


# 지역유형별 집계
sdot_type = df_sdot.groupby('지역').agg(
    방문자수_합=('방문자수', 'sum'),
    센서수=('시리얼', 'nunique'),
    레코드수=('방문자수', 'count')
).reset_index()

sdot_type['일평균_방문자'] = (sdot_type['방문자수_합'] / total_days).round(0)
sdot_type['센서당_일평균방문자'] = (sdot_type['일평균_방문자'] / sdot_type['센서수']).round(0)
sdot_type = sdot_type.sort_values('일평균_방문자', ascending=False)

# 유형 한글 매핑
TYPE_MAP = {
    'main_street': '주요 거리',
    'traditional_markets': '전통시장',
    'parks': '공원',
    'commercial_area': '상업지역',
    'residential_area': '주거지역',
    'public_facilities': '공공시설'
}
sdot_type['유형_한글'] = sdot_type['지역'].map(TYPE_MAP)

print('=' * 65)
print('S-DoT 지역유형별 유동인구')
print('=' * 65)
print(sdot_type[['유형_한글', '일평균_방문자', '센서수', '센서당_일평균방문자']].to_string(index=False))

# 관광 vs 거주 비교
tourism_types = ['main_street', 'traditional_markets', 'commercial_area']
residential_types = ['residential_area']

tourism_avg = sdot_type[sdot_type['지역'].isin(tourism_types)]['일평균_방문자'].sum()
residential_avg = sdot_type[sdot_type['지역'].isin(residential_types)]['일평균_방문자'].sum()

print(f'\n관광/상업 유형 일평균 합계: {tourism_avg:,.0f}명')
print(f'주거 유형 일평균: {residential_avg:,.0f}명')
if residential_avg > 0:
    print(f'관광/상업 대비 주거 비율: {tourism_avg / residential_avg:.1f}배')

S-DoT 지역유형별 유동인구
유형_한글  일평균_방문자  센서수  센서당_일평균방문자
 전통시장 757250.0   50     15145.0
주요 거리 199981.0   35      5714.0
 공공시설  62472.0    6     10412.0
   공원   7284.0    5      1457.0
 상업지역   3217.0    1      3217.0
 주거지역   1577.0    1      1577.0

관광/상업 유형 일평균 합계: 960,448명
주거 유형 일평균: 1,577명
관광/상업 대비 주거 비율: 609.0배


In [ ]:
# S-DoT 시간대별 패턴 분석


# 시간대별 전체 유동인구
sdot_hourly = df_sdot.groupby('시간').agg(
    방문자수_합=('방문자수', 'sum')
).reset_index()
sdot_hourly['일평균_방문자'] = (sdot_hourly['방문자수_합'] / total_days).round(0)

print('=' * 50)
print('시간대별 유동인구 추이 (10~22시)')
print('=' * 50)
print(sdot_hourly[['시간', '일평균_방문자']].to_string(index=False))

peak_hour = sdot_hourly.loc[sdot_hourly['일평균_방문자'].idxmax()]
print(f'\n피크타임: {int(peak_hour["시간"])}시 (일평균 {peak_hour["일평균_방문자"]:,.0f}명)')

# 주요 관광 동별 시간대 비교
tourist_dongs = ['Myeong-dong', 'Itaewon2-dong', 'Apgujeong-dong', 'Sinsa-dong', 'Hoehyeon-dong']
dong_labels = {'Myeong-dong': '명동', 'Itaewon2-dong': '이태원', 'Apgujeong-dong': '압구정',
               'Sinsa-dong': '신사', 'Hoehyeon-dong': '회현(남대문)'}

print(f'\n{"=" * 50}')
print('주요 관광지 시간대별 유동인구')
print('=' * 50)

for dong in tourist_dongs:
    dong_data = df_sdot[df_sdot['행정동'] == dong]
    if len(dong_data) == 0:
        continue
    hourly = dong_data.groupby('시간')['방문자수'].sum() / total_days
    peak = hourly.idxmax()
    label = dong_labels.get(dong, dong)
    print(f'  {label}: 피크 {peak}시 (일평균 {hourly.max():.0f}명), 전체 일평균 {hourly.sum():.0f}명')

시간대별 유동인구 추이 (10~22시)
 시간  일평균_방문자
 10  75264.0
 11  81163.0
 12  88895.0
 13  85574.0
 14  83726.0
 15  84756.0
 16  84998.0
 17  86612.0
 18  87023.0
 19  80526.0
 20  72278.0
 21  64698.0
 22  56266.0

피크타임: 12시 (일평균 88,895명)

주요 관광지 시간대별 유동인구


  명동: 피크 20시 (일평균 3814명), 전체 일평균 42222명


  이태원: 피크 18시 (일평균 21명), 전체 일평균 214명


  압구정: 피크 18시 (일평균 319명), 전체 일평균 3476명


  신사: 피크 18시 (일평균 761명), 전체 일평균 6424명


  회현(남대문): 피크 12시 (일평균 1175명), 전체 일평균 8266명


In [ ]:
# S-DoT 동 단위 상세 분석


# 행정동별 유동인구 집계
sdot_dong = df_sdot.groupby(['자치구_한글', '행정동']).agg(
    방문자수_합=('방문자수', 'sum'),
    센서수=('시리얼', 'nunique')
).reset_index()

sdot_dong.rename(columns={'자치구_한글': '자치구'}, inplace=True)
sdot_dong['일평균_방문자'] = (sdot_dong['방문자수_합'] / total_days).round(0)
sdot_dong['센서당_일평균방문자'] = (sdot_dong['일평균_방문자'] / sdot_dong['센서수']).round(0)
sdot_dong = sdot_dong.sort_values('일평균_방문자', ascending=False)

# 한글 동명 매핑
DONG_MAP = {
    'Myeong-dong': '명동', 'Hoehyeon-dong': '회현동(남대문)', 'Sinsa-dong': '신사동',
    'Apgujeong-dong': '압구정동', 'Itaewon2-dong': '이태원2동', 'Mangwon-dong1': '망원동',
    'Gwanghui-dong': '광희동(DDP)', 'Samcheong-dong': '삼청동', 'Gahoe-dong': '가회동(북촌)',
    'Ihwa-dong': '이화동(이화벽화마을)', 'Sindang-dong': '신당동', 'Yeoksam1-dong': '역삼1동',
    'Seongsu1ga2-dong': '성수1가2동', 'Seocho3-dong': '서초3동', 'Seocho4-dong': '서초4동',
    'Daechi4-dong': '대치4동', 'Jamsil6-dong': '잠실6동', 'Banpo3-dong': '반포3동',
    'Seongsu1ga1-dong': '성수1가1동', 'Hwayang-dong': '화양동(건대)',
    'Buam-dong': '부암동', 'Suyu3-dong': '수유3동', 'Nakseongdae-dong': '낙성대동',
    'Nokbeon-dong': '녹번동', 'Hongje3-dong': '홍제3동', 'Gonghang-dong': '공항동',
    'Gayang-dong1': '가양동', 'Deungchon-dong1': '등촌동', 'Sinjeong4-dong': '신정4동',
    'Cheonyeon-dong': '천연동', 'Chang1-dong': '창1동', 'Beon3-dong': '번3동',
    'Daejo-dong': '대조동', 'Cheonho1-dong': '천호1동', 'Amsa3-dong': '암사3동',
    'Guui1-dong': '구의1동', 'Gwangjang-dong': '광장동', 'Yangjae1-dong': '양재1동',
    'Seongnae1-dong': '성내1동', 'Banghwa1-dong': '방화1동', 'Banghwa3-dong': '방화3동'
}
sdot_dong['행정동_한글'] = sdot_dong['행정동'].map(DONG_MAP).fillna(sdot_dong['행정동'])

print('=' * 75)
print('S-DoT 행정동별 유동인구 TOP 20')
print('=' * 75)
top20 = sdot_dong.head(20)
print(top20[['자치구', '행정동_한글', '일평균_방문자', '센서수', '센서당_일평균방문자']].to_string(index=False))

# 보고서의 관광특구/발달상권 매칭
print(f'\n{"=" * 75}')
print('관광특구 매칭 동별 유동인구')
print('=' * 75)
tourism_dongs = ['Myeong-dong', 'Hoehyeon-dong', 'Itaewon2-dong', 'Apgujeong-dong', 'Sinsa-dong',
                 'Samcheong-dong', 'Gahoe-dong', 'Gwanghui-dong']
tourism_data = sdot_dong[sdot_dong['행정동'].isin(tourism_dongs)].sort_values('일평균_방문자', ascending=False)
print(tourism_data[['자치구', '행정동_한글', '일평균_방문자', '센서당_일평균방문자']].to_string(index=False))

S-DoT 행정동별 유동인구 TOP 20
자치구              행정동_한글  일평균_방문자  센서수  센서당_일평균방문자
 중구                  명동  42222.0    2     21111.0
송파구      Jamsilbon-dong  39459.0    2     19730.0
양천구    Sinwol1(il)-dong  38754.0    3     12918.0
 중구            광희동(DDP)  38595.0    1     38595.0
강북구            Mia-dong  37604.0    2     18802.0
관악구         Sinwon-dong  37242.0    2     18621.0
강동구      Amsa1(il)-dong  35455.0    3     11818.0
구로구     Gu-ro4(sa)-dong  31844.0    3     10615.0
도봉구      Chang2(i)-dong  26463.0    3      8821.0
동작구     Sadang2(i)-dong  25768.0    2     12884.0
동작구   Sang-do1(il)-dong  23441.0    2     11720.0
구로구    Gocheok2(i)-dong  22328.0    2     11164.0
금천구    Doksan4(sa)-dong  19845.0    2      9922.0
양천구        Sinwol1-dong  19216.0    3      6405.0
양천구      Mok3(sam)-dong  17893.0    1     17893.0
동작구   Sang-do4(sa)-dong  17862.0    1     17862.0
노원구 Gongneung1(il)-dong  17443.0    2      8722.0
강동구          Amsa1-dong  17385.0    3      5795.0
마포구         Seogyo-dong  17

In [ ]:
# 복합점수 계산 (외국인 + 유동인구)


# 외국인 데이터에서 자치구별 일평균 외국인 추출
foreigner_gu = df_foreigner[['자치구', '일평균_총외국인']].copy()
foreigner_gu.rename(columns={'일평균_총외국인': '일평균_외국인'}, inplace=True)

# S-DoT 자치구별 일평균 방문자
sdot_gu_merge = sdot_gu[['자치구', '일평균_방문자', '센서수', '센서당_일평균방문자']].copy()

# 병합
merged = pd.merge(foreigner_gu, sdot_gu_merge, on='자치구', how='outer')

# 복합점수 계산 (양쪽 데이터 모두 있는 경우)
df_score = merged.dropna(subset=['일평균_외국인', '일평균_방문자']).copy()

# Min-Max 정규화
for col, new_col in [('일평균_외국인', '외국인_정규화'), ('일평균_방문자', '유동량_정규화')]:
    min_val = df_score[col].min()
    max_val = df_score[col].max()
    df_score[new_col] = ((df_score[col] - min_val) / (max_val - min_val + 1e-10)).round(4)

# 복합점수 = 외국인_정규화 + 유동량_정규화 (0~2 범위)
df_score['복합점수'] = (df_score['외국인_정규화'] + df_score['유동량_정규화']).round(4)

# Hub/Spoke 분류 (상위 30% = Hub)
threshold = df_score['복합점수'].quantile(0.70)
df_score['분류'] = df_score['복합점수'].apply(lambda x: 'Hub' if x >= threshold else 'Spoke')

df_score = df_score.sort_values('복합점수', ascending=False)

print('=' * 85)
print('복합점수 분석 결과 (외국인 밀집도 + 유동인구)')
print('=' * 85)
print(f'Hub/Spoke 기준 복합점수: {threshold:.4f} (상위 30%)')
print()
display_cols = ['자치구', '일평균_외국인', '일평균_방문자', '외국인_정규화', '유동량_정규화', '복합점수', '분류']
print(df_score[display_cols].to_string(index=False))

print(f'\nHub 자치구: {df_score[df_score["분류"]=="Hub"]["자치구"].tolist()}')
print(f'Spoke 자치구: {df_score[df_score["분류"]=="Spoke"]["자치구"].tolist()}')

복합점수 분석 결과 (외국인 밀집도 + 유동인구)
Hub/Spoke 기준 복합점수: 0.5969 (상위 30%)

 자치구  일평균_외국인  일평균_방문자  외국인_정규화  유동량_정규화   복합점수    분류
  중구   898026  93822.0   1.0000   0.8865 1.8865   Hub
 강남구   418729 105765.0   0.4640   1.0000 1.4640   Hub
 동작구    25357 100824.0   0.0241   0.9531 0.9772   Hub
 양천구     7277  90469.0   0.0039   0.8547 0.8586   Hub
 구로구    65215  81066.0   0.0687   0.7653 0.8340   Hub
 마포구   372379  42447.0   0.4122   0.3984 0.8106   Hub
 도봉구     3791  63945.0   0.0000   0.6026 0.6026   Hub
 강동구    14606  60684.0   0.0121   0.5716 0.5837 Spoke
 종로구   330441  21381.0   0.3653   0.1982 0.5635 Spoke
 광진구    44670  54661.0   0.0457   0.5144 0.5601 Spoke
 송파구   111388  42900.0   0.1203   0.4027 0.5230 Spoke
 금천구    26837  50587.0   0.0258   0.4757 0.5015 Spoke
 강북구     9519  52272.0   0.0064   0.4917 0.4981 Spoke
 강서구   161788  30114.0   0.1767   0.2812 0.4579 Spoke
 관악구    36688  42892.0   0.0368   0.4026 0.4394 Spoke
 노원구     8041  41165.0   0.0048   0.3862 0.3910 Spoke
서대문구   287386   51

In [ ]:
# 결과 저장


os.makedirs(PROCESSED, exist_ok=True)

# 1. 복합분석 결과
save_path_1 = os.path.join(PROCESSED, '복합분석_외국인_유동인구.csv')
df_score.to_csv(save_path_1, index=False, encoding='utf-8-sig')
print(f'[저장] {save_path_1}')

# 2. S-DoT 자치구별 유동인구
save_path_2 = os.path.join(PROCESSED, 'S-DoT_자치구별_유동인구.csv')
sdot_gu.to_csv(save_path_2, index=False, encoding='utf-8-sig')
print(f'[저장] {save_path_2}')

# 3. S-DoT 지역유형별 유동인구
save_path_3 = os.path.join(PROCESSED, 'S-DoT_지역유형별_유동인구.csv')
sdot_type.to_csv(save_path_3, index=False, encoding='utf-8-sig')
print(f'[저장] {save_path_3}')

# 4. S-DoT 동별 유동인구 TOP
save_path_4 = os.path.join(PROCESSED, 'S-DoT_동별_유동인구_TOP.csv')
sdot_dong.head(30).to_csv(save_path_4, index=False, encoding='utf-8-sig')
print(f'[저장] {save_path_4}')

print(f'\n모든 결과 저장 완료!')

[저장] /Users/yu_seok/Documents/workspace/01_현재진행/01_nbCamp/nb_Project/Why-pi/06_analysis/01_foreigner/02_processed_data/복합분석_외국인_유동인구.csv
[저장] /Users/yu_seok/Documents/workspace/01_현재진행/01_nbCamp/nb_Project/Why-pi/06_analysis/01_foreigner/02_processed_data/S-DoT_자치구별_유동인구.csv
[저장] /Users/yu_seok/Documents/workspace/01_현재진행/01_nbCamp/nb_Project/Why-pi/06_analysis/01_foreigner/02_processed_data/S-DoT_지역유형별_유동인구.csv
[저장] /Users/yu_seok/Documents/workspace/01_현재진행/01_nbCamp/nb_Project/Why-pi/06_analysis/01_foreigner/02_processed_data/S-DoT_동별_유동인구_TOP.csv

모든 결과 저장 완료!
